# Stage 4 — End-to-end 파이프라인 (PDF → view → element → Donut → JSON)

도면 PDF 1장을 넣으면 전체 파이프라인을 통과시켜 구조화 JSON 을 조립합니다.

**큰 그림**: 도면은 한 번에 읽기 어렵습니다. 그래서 **"점점 좁혀가며 본다"** 는 전략을 씁니다.
도면 전체 → 뷰(부분 도면) → 요소(치수·기호) 순으로 잘라 들어간 뒤, 마지막에 요소 안의
**글자(값)** 를 읽습니다. 즉 **"어디에 있나(검출)"는 YOLO**, **"뭐라고 적혔나(인식)"는 Donut** 이
나눠 맡습니다.

```
┌──────────────────────────────────────────────────────────────────────┐
│  입력:  도면 PDF 1장                                                    │
└──────────────────────────────────────────────────────────────────────┘
        │
        │ ① 래스터화  (PyMuPDF · 300 DPI)
        │    벡터 PDF → 픽셀 이미지 1장. 이후 단계는 모두 이미지를 봄
        ▼
   page.png  ── 도면 전체 한 장
        │
        │ ② 뷰 검출  ──  YOLO (detect, AABB = 수평 사각형 박스)
        │    "도면 안에 정면도·평면도·표제란 같은 '뷰'가 어디 있나?"
        │    뷰는 똑바로 놓여 있어 회전 없는 박스(AABB)로 충분
        ▼
   뷰 크롭들  ── 예: 5개  (정면도 / 측면도 / 표제란 …)
        │
        │   ┌─ 각 뷰 크롭마다 ③~④ 반복 ───────────────────────────┐
        │   │                                                       │
        │   │ ③ 요소 검출  ──  YOLO (OBB = 회전 가능한 박스) + rectify
        │   │    "이 뷰 안에 치수·공차·거칠기 같은 '요소'가 어디 있나?"
        │   │    치수선은 기울어 있기도 함 → OBB 로 잡아
        │   │    똑바로 펴서(rectify) 잘라냄 → Donut 이 읽기 쉬워짐
        │   │      ▼
        │   │   요소 크롭들 (+ type)
        │   │      ── 크롭마다 YOLO 가 매긴 종류(dimension / gdt / … )
        │   │      │
        │   │ ④ 요소 인식  ──  Donut  (크롭 1장 → 텍스트 값)
        │   │    "이 요소 크롭에 적힌 값이 정확히 뭐냐?"
        │   │    예) "Ø65" ,  "Ra 1.6" ,  "M8" …
        │   │      ▼
        │   │   {type, value, box}  ── 종류 + 읽은 값 + 위치
        │   └───────────────────────────────────────────────────────┘
        │
        │ ⑤ 조립(assemble)
        │    뷰별로 요소들을 묶고, 요소 좌표를 뷰 기준 → page(전체) 기준으로 환산
        ▼
┌──────────────────────────────────────────────────────────────────────┐
│  출력:  구조화 JSON                                                     │
│  {                                                                     │
│    "views": [                                                          │
│      { "view_index": 0,                                                │
│        "box": [...],                       ← 뷰의 page 상 위치          │
│        "elements": [                                                   │
│          {"type": "dimension",      "value": "Ø65",    "box": [...]},  │
│          {"type": "surface_finish", "value": "Ra 1.6", "box": [...]}   │
│        ] }                                                             │
│    ]                                                                   │
│  }                                                                     │
└──────────────────────────────────────────────────────────────────────┘
```

> **용어**
> - **AABB** (Axis-Aligned Bounding Box): 회전 없는 수평 직사각형 박스. 뷰처럼 똑바로 놓인 영역에 사용.
> - **OBB** (Oriented Bounding Box): 기울기(각도)를 가진 박스. 회전된 치수선·기호를 정확히 감쌈.
> - **rectify**: OBB 로 잡은 기울어진 요소를 회전 보정해 **똑바로 편** 크롭. 글자가 수평이라 인식 정확도↑.
> - **검출(YOLO) vs 인식(Donut)**: YOLO 는 "**어디에** 무엇이 있나(위치+종류)", Donut 은 "거기 **뭐라고** 적혔나(값)".

**커널**: 전 과정을 **단일 커널 `kardi_env`** 로 실행합니다. `kardi_env` 에 YOLO(ultralytics)와
Donut(transformers/torch) 의존성이 모두 설치돼 있어 **커널 전환이 필요 없습니다**.
- **Part A**: 검출·크롭 → `data/_pipeline_tmp/` 에 크롭 + `meta.json` 저장
- **Part B**: 크롭을 Donut 으로 읽어 값 채우고 최종 JSON 조립

→ Part A → Part B 를 같은 커널에서 위→아래로 이어서 실행하면 됩니다.
(`meta.json`/크롭은 디버깅용 중간 산출물.)

## Part A — 검출·크롭 (커널: **kardi_env**)

In [ ]:
# ── 설정 ─────────────────────────────────────────────────────────
from pathlib import Path
import json, sys
sys.path.append("detection")                     # crop_utils
from crop_utils import save_crops_from_result
from ultralytics import YOLO

PDF_PATH   = Path("../data/raw_pdf/A1370954730.pdf")  # ← held-out 도면으로 바꾸세요
TMP        = Path("../data/_pipeline_tmp"); TMP.mkdir(parents=True, exist_ok=True)
# ultralytics 8.4.67 이 task 를 runs/{task}/runs/{name} 으로 중첩 저장함 (train_*.ipynb 산출물)
VIEW_PT    = "detection/view/runs/detect/runs/view/weights/best.pt"      # view: detect task
ELEM_PT    = "detection/element/runs/obb/runs/element/weights/best.pt"   # element: obb task
DPI        = 300
VIEW_PAD   = 4     # view 크롭 여유 픽셀 (element→page 좌표 변환에 사용)
ELEM_PAD   = 2     # element 정렬 크롭 여유 픽셀

In [13]:
# ── 1) PDF → page.png ────────────────────────────────────────────
import fitz
doc = fitz.open(PDF_PATH)
pix = doc[0].get_pixmap(matrix=fitz.Matrix(DPI/72, DPI/72), alpha=False)
page_png = TMP / f"{PDF_PATH.stem}.png"
pix.save(page_png); doc.close()
print("page:", page_png)

page: ../data/_pipeline_tmp/A1370954730.png


In [14]:
# ── 2) view 검출 → view 크롭 ─────────────────────────────────────
# train_view.ipynb 에서 학습한 모델(VIEW_PT : best.pt) 사용
view_model = YOLO(VIEW_PT)
vr = view_model.predict(page_png, imgsz=1280, conf=0.25, verbose=False)[0]
view_meta = save_crops_from_result(vr, TMP / "views", PDF_PATH.stem, pad=VIEW_PAD)
print(f"view {len(view_meta)}개")

view 5개


In [15]:
# ── 3) 각 view 크롭 → element 검출 → rectify 크롭 ────────────────
elem_model = YOLO(ELEM_PT)  # 요소 YOLO 모델 로드

def _to_page(xy, ox, oy):
    # element box(view 기준) → page 기준 좌표로 변환. OBB, AABB 모두 지원
    if xy and isinstance(xy[0], (list, tuple)):  # OBB polygon
        return [[float(x) + ox, float(y) + oy] for x, y in xy]
    return [xy[0] + ox, xy[1] + oy, xy[2] + ox, xy[3] + oy]  # AABB

records = []   # 뷰/요소 메타데이터를 저장
for vi, vm in enumerate(view_meta):
    er = elem_model.predict(vm["path"], imgsz=1024, conf=0.25, verbose=False)[0]  # 요소 검출
    elems = save_crops_from_result(
        er,
        TMP / "elements" / f"view{vi}",
        f"v{vi}",
        pad=ELEM_PAD
    )  # 요소 크롭 저장

    # view 크롭의 page 상 기준점 (pad까지 고려, 음수 방지)
    ox, oy = max(0.0, vm["xy"][0] - VIEW_PAD), max(0.0, vm["xy"][1] - VIEW_PAD)
    for e in elems:
        e["xy_page"] = _to_page(e["xy"], ox, oy)  # 요소 box를 page 좌표로 변환
    records.append({"view_index": vi, "view": vm, "elements": elems})

meta_path = TMP / "meta.json"
json.dump(records, open(meta_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)  # 결과 저장
n_elem = sum(len(r["elements"]) for r in records)
print(f"element 총 {n_elem}개 → {meta_path}")  # 총 요소 개수 출력

element 총 47개 → ../data/_pipeline_tmp/meta.json


## Part B — Donut 값 추론 + JSON 조립 (커널: **kardi_env**)

In [16]:
# ── Donut 로드 + 추론 함수 ───────────────────────────────────────
import json, sys, torch
from pathlib import Path
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel
sys.path.append("detection")                     # 공용 토큰 I/O (학습 노트북과 동일 로직)
from donut_utils import decode_donut_output

TMP        = Path("../data/_pipeline_tmp")
CKPT       = "../checkpoints_elements/final"
TASK       = "<s_element>"
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
proc       = DonutProcessor.from_pretrained(CKPT)
emodel     = VisionEncoderDecoderModel.from_pretrained(CKPT).to(device).eval()

@torch.inference_mode()
def read_value(img_path):
    image = Image.open(img_path).convert("RGB")
    pv  = proc(image, return_tensors="pt").pixel_values.to(device)
    dec = proc.tokenizer(TASK, add_special_tokens=False, return_tensors="pt").input_ids.to(device)
    out = emodel.generate(pv, decoder_input_ids=dec, max_new_tokens=128,
                          pad_token_id=proc.tokenizer.pad_token_id,
                          eos_token_id=proc.tokenizer.eos_token_id, use_cache=True)
    seq = proc.batch_decode(out, skip_special_tokens=False)[0]
    # 특수 토큰(eos/pad/bos)+task 제거 후 dict 파싱 (공용 헬퍼 — strip 집합을 한 곳에서 관리)
    return decode_donut_output(seq, proc.tokenizer, TASK)

OSError: Can't load image processor for '../checkpoints_elements/final'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure '../checkpoints_elements/final' is the correct path to a directory containing a preprocessor_config.json file

In [ ]:
# ── 조립: meta.json 의 element 크롭마다 값 추론 → 최종 JSON ──────
records = json.load(open(TMP / "meta.json", encoding="utf-8"))
assembled = {"source": str(TMP), "views": []}
for r in records:
    view_out = {"view_index": r["view_index"], "box": r["view"]["xy"], "elements": []}
    for e in r["elements"]:
        parsed = read_value(e["path"])        # {"<type>": "<value>"} 기대
        # YOLO 가 분류한 type 으로 보정: 그 키를 우선, 비어 있으면 첫 값, dict 가 아니면 그대로
        if isinstance(parsed, dict):
            value = parsed.get(e["name"]) or next(iter(parsed.values()), parsed)
        else:
            value = parsed
        view_out["elements"].append({"type": e["name"], "value": value,
                                      "conf": round(e["conf"], 3),
                                      "box": e["xy_page"],       # 페이지 좌표 (view box 와 동일 좌표계)
                                      "box_in_view": e["xy"]})   # 디버그용: view-크롭 내부 좌표
    assembled["views"].append(view_out)

print(json.dumps(assembled, ensure_ascii=False, indent=2)[:2000])
out_path = TMP / "result.json"
json.dump(assembled, open(out_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("\n저장:", out_path)